In [0]:
%sql
SHOW TABLES IN workspace.gold;

database,tableName,isTemporary
gold,dim_clientes,false
gold,dim_produtos,false
gold,dim_tempo,false
gold,dim_vendedores,false
gold,fato_itens_pedido,false
gold,fato_pagamentos,false
gold,fato_pedidos,false


## Catálogo de Dados — Camada Gold

Este notebook documenta o modelo analítico desenvolvido
para o projeto Brazilian E-Commerce da Olist.

O catálogo registra as descrições das tabelas e colunas,
seus tipos de dados, domínios de valores e a origem
das informações.

A documentação é armazenada no Unity Catalog do Databricks,
facilitando a governança, rastreabilidade e compreensão
dos dados disponibilizados para análise.

In [0]:
%sql

-- Dimensão de clientes
COMMENT ON TABLE workspace.gold.dim_clientes IS
'Dimensão analítica contendo informações geográficas dos clientes da Olist. Origem: workspace.silver.customers. Granularidade: um registro por customer_id.';

-- Dimensão de produtos
COMMENT ON TABLE workspace.gold.dim_produtos IS
'Dimensão analítica contendo características físicas e categorias dos produtos comercializados. Origem: workspace.silver.products e workspace.silver.category_translation. Granularidade: um registro por product_id.';

-- Dimensão de tempo
COMMENT ON TABLE workspace.gold.dim_tempo IS
'Dimensão calendário utilizada para análises temporais das vendas e pedidos. Gerada a partir do intervalo de datas de compra dos pedidos. Granularidade: um registro por dia.';

-- Dimensão de vendedores
COMMENT ON TABLE workspace.gold.dim_vendedores IS
'Dimensão analítica contendo informações geográficas dos vendedores da Olist. Origem: workspace.silver.sellers. Granularidade: um registro por seller_id.';

-- Fato de pedidos
COMMENT ON TABLE workspace.gold.fato_pedidos IS
'Tabela fato contendo informações de compra, entrega, atraso e satisfação dos consumidores. Origem: workspace.silver.orders e workspace.silver.order_reviews. Granularidade: um registro por pedido.';

-- Fato de itens de pedido
COMMENT ON TABLE workspace.gold.fato_itens_pedido IS
'Tabela fato contendo informações comerciais dos itens vendidos, incluindo produto, vendedor, preço e frete. Origem: workspace.silver.order_items e workspace.gold.fato_pedidos. Granularidade: um registro por item de pedido.';

-- Fato de pagamentos
COMMENT ON TABLE workspace.gold.fato_pagamentos IS
'Tabela fato contendo valores financeiros, meios de pagamento e quantidade de parcelas. Origem: workspace.silver.order_payments e workspace.gold.fato_pedidos. Granularidade: um registro por registro de pagamento.';

### 1. Validação da documentação das tabelas

Verificação dos comentários registrados nas tabelas
da camada Gold por meio dos metadados do Unity Catalog.

In [0]:
%sql

SELECT
    table_name,
    comment

FROM workspace.information_schema.tables

WHERE table_schema = 'gold'
  AND table_catalog = 'workspace'

ORDER BY table_name;

table_name,comment
dim_clientes,Dimensão analítica contendo informações geográficas dos clientes da Olist. Origem: workspace.silver.customers. Granularidade: um registro por customer_id.
dim_produtos,Dimensão analítica contendo características físicas e categorias dos produtos comercializados. Origem: workspace.silver.products e workspace.silver.category_translation. Granularidade: um registro por product_id.
dim_tempo,Dimensão calendário utilizada para análises temporais das vendas e pedidos. Gerada a partir do intervalo de datas de compra dos pedidos. Granularidade: um registro por dia.
dim_vendedores,Dimensão analítica contendo informações geográficas dos vendedores da Olist. Origem: workspace.silver.sellers. Granularidade: um registro por seller_id.
fato_itens_pedido,"Tabela fato contendo informações comerciais dos itens vendidos, incluindo produto, vendedor, preço e frete. Origem: workspace.silver.order_items e workspace.gold.fato_pedidos. Granularidade: um registro por item de pedido."
fato_pagamentos,"Tabela fato contendo valores financeiros, meios de pagamento e quantidade de parcelas. Origem: workspace.silver.order_payments e workspace.gold.fato_pedidos. Granularidade: um registro por registro de pagamento."
fato_pedidos,"Tabela fato contendo informações de compra, entrega, atraso e satisfação dos consumidores. Origem: workspace.silver.orders e workspace.silver.order_reviews. Granularidade: um registro por pedido."


### 2. Levantamento dos atributos das tabelas Gold

Consulta aos metadados do Unity Catalog para identificar
os atributos existentes em cada tabela, seus respectivos
tipos de dados e sua organização no modelo analítico.

Esse levantamento servirá de base para a documentação
individual das colunas, incluindo descrições, domínios
de valores e linhagem dos dados.

In [0]:
%sql

SELECT
    table_name,
    ordinal_position,
    column_name,
    data_type,
    is_nullable,
    comment

FROM workspace.information_schema.columns

WHERE table_schema = 'gold'
  AND table_catalog = 'workspace'

ORDER BY
    table_name,
    ordinal_position;

table_name,ordinal_position,column_name,data_type,is_nullable,comment
dim_clientes,0,customer_id,STRING,YES,null
dim_clientes,1,customer_unique_id,STRING,YES,null
dim_clientes,2,cep_prefixo,STRING,YES,null
dim_clientes,3,cidade,STRING,YES,null
dim_clientes,4,estado,STRING,YES,null
dim_produtos,0,product_id,STRING,YES,null
dim_produtos,1,categoria_original,STRING,YES,null
dim_produtos,2,categoria_analitica,STRING,YES,null
dim_produtos,3,tamanho_nome,INT,YES,null
dim_produtos,4,tamanho_descricao,INT,YES,null


#### 2.1 Inventário de atributos por tabela

Levantamento consolidado dos atributos das tabelas Gold,
organizados por tabela e acompanhados dos respectivos
tipos de dados.

Este inventário será utilizado como referência para
a documentação dos campos no Unity Catalog.

In [0]:
%sql

SELECT
    table_name,

    concat_ws(
        '\n',
        collect_list(
            concat(
                column_name,
                ' (',
                data_type,
                ')'
            )
        )
    ) AS atributos

FROM workspace.information_schema.columns

WHERE table_schema = 'gold'
  AND table_catalog = 'workspace'

GROUP BY table_name

ORDER BY table_name;

table_name,atributos
dim_clientes,customer_id (STRING) customer_unique_id (STRING) cep_prefixo (STRING) cidade (STRING) estado (STRING)
dim_produtos,product_id (STRING) categoria_original (STRING) categoria_analitica (STRING) tamanho_nome (INT) tamanho_descricao (INT) quantidade_fotos (INT) peso_gramas (DECIMAL) comprimento_cm (DECIMAL) altura_cm (DECIMAL) largura_cm (DECIMAL)
dim_tempo,data_id (INT) data (DATE) ano (INT) mes (INT) trimestre (INT) dia (INT) dia_semana (INT) ano_mes (STRING)
dim_vendedores,seller_id (STRING) cep_prefixo (STRING) cidade (STRING) estado (STRING)
fato_itens_pedido,order_id (STRING) order_item_id (INT) product_id (STRING) seller_id (STRING) data_id (INT) valor_produto (DECIMAL) valor_frete (DECIMAL) valor_total_item (DECIMAL)
fato_pagamentos,order_id (STRING) payment_sequential (INT) data_id (INT) payment_type (STRING) payment_installments (INT) payment_value (DECIMAL)
fato_pedidos,order_id (STRING) customer_id (STRING) data_id (INT) order_status (STRING) order_purchase_timestamp (TIMESTAMP) order_delivered_customer_date (TIMESTAMP) order_estimated_delivery_date (TIMESTAMP) tempo_entrega_dias (DOUBLE) atraso_dias (INT) status_prazo (STRING) nota_media_avaliacao (DOUBLE) quantidade_avaliacoes (LONG) quantidade_comentarios (LONG)


### 3. Documentação dos atributos no Unity Catalog

Nesta etapa, os atributos das tabelas Gold serão documentados
individualmente, registrando seu significado de negócio,
domínio de valores esperado e origem dos dados.

A documentação será persistida no Unity Catalog por meio
de comandos SQL, permitindo consultar os metadados
diretamente na plataforma Databricks.

In [0]:
%python
# Recupera automaticamente as colunas das tabelas Gold
df_colunas = spark.sql("""
    SELECT
        table_name,
        column_name,
        data_type
    FROM workspace.information_schema.columns
    WHERE table_schema = 'gold'
      AND table_catalog = 'workspace'
    ORDER BY table_name, ordinal_position
""")

# Organiza os atributos por tabela
tabelas = {}

for linha in df_colunas.collect():

    nome_tabela = linha["table_name"]

    if nome_tabela not in tabelas:
        tabelas[nome_tabela] = []

    tabelas[nome_tabela].append({
        "coluna": linha["column_name"],
        "tipo": linha["data_type"]
    })

# Exibe o inventário completo
for tabela, colunas in tabelas.items():

    print(f"\nTABELA: {tabela}")

    for coluna in colunas:
        print(
            f"  {coluna['coluna']} "
            f"({coluna['tipo']})"
        )


TABELA: dim_clientes
  customer_id (STRING)
  customer_unique_id (STRING)
  cep_prefixo (STRING)
  cidade (STRING)
  estado (STRING)

TABELA: dim_produtos
  product_id (STRING)
  categoria_original (STRING)
  categoria_analitica (STRING)
  tamanho_nome (INT)
  tamanho_descricao (INT)
  quantidade_fotos (INT)
  peso_gramas (DECIMAL)
  comprimento_cm (DECIMAL)
  altura_cm (DECIMAL)
  largura_cm (DECIMAL)

TABELA: dim_tempo
  data_id (INT)
  data (DATE)
  ano (INT)
  mes (INT)
  trimestre (INT)
  dia (INT)
  dia_semana (INT)
  ano_mes (STRING)

TABELA: dim_vendedores
  seller_id (STRING)
  cep_prefixo (STRING)
  cidade (STRING)
  estado (STRING)

TABELA: fato_itens_pedido
  order_id (STRING)
  order_item_id (INT)
  product_id (STRING)
  seller_id (STRING)
  data_id (INT)
  valor_produto (DECIMAL)
  valor_frete (DECIMAL)
  valor_total_item (DECIMAL)

TABELA: fato_pagamentos
  order_id (STRING)
  payment_sequential (INT)
  data_id (INT)
  payment_type (STRING)
  payment_installments (INT)


#### 3.1 Definição do dicionário de dados

Criação de um dicionário contendo as descrições dos atributos
das sete tabelas da camada Gold.

Cada descrição contempla o significado do atributo, seu domínio
de valores esperado e sua origem ou transformação.

O dicionário será utilizado para registrar os metadados
diretamente no Unity Catalog.

In [0]:
%python
# Dicionário de dados das tabelas Gold
# Cada atributo recebe uma descrição individual para o Unity Catalog.

documentacao = {

    # =====================================================
    # DIMENSÃO DE CLIENTES
    # =====================================================

    "dim_clientes": {

        "customer_id":
        "Identificador do cliente associado ao pedido. Origem: silver.customers.customer_id. Dominio: identificador textual.",

        "customer_unique_id":
        "Identificador unico do consumidor, permitindo reconhecer compras realizadas pelo mesmo consumidor. Origem: silver.customers.customer_unique_id. Dominio: identificador textual.",

        "cep_prefixo":
        "Prefixo de cinco digitos do CEP do cliente. Origem: customer_zip_code_prefix. Dominio esperado: cinco digitos.",

        "cidade":
        "Cidade de residencia do cliente. Origem: customer_city. Dominio: nome textual de municipio brasileiro.",

        "estado":
        "Unidade federativa do cliente. Origem: customer_state. Dominio: siglas das 27 unidades federativas brasileiras."

    },

    # =====================================================
    # DIMENSÃO DE PRODUTOS
    # =====================================================

    "dim_produtos": {

        "product_id":
        "Identificador unico do produto. Origem: silver.products.product_id. Dominio: identificador textual.",

        "categoria_original":
        "Categoria original do produto em portugues. Origem: silver.products.product_category_name. Dominio: categorias existentes no dataset Olist ou categoria nao informada.",

        "categoria_analitica":
        "Categoria padronizada utilizada nas analises. Origem: categoria original e tabela silver.category_translation. Dominio: categorias traduzidas ou categoria nao informada.",

        "tamanho_nome":
        "Quantidade de caracteres do nome do produto. Origem: product_name_lenght. Dominio esperado: inteiro nao negativo; pode ser nulo.",

        "tamanho_descricao":
        "Quantidade de caracteres da descricao do produto. Origem: product_description_lenght. Dominio esperado: inteiro nao negativo; pode ser nulo.",

        "quantidade_fotos":
        "Quantidade de fotografias cadastradas para o produto. Origem: product_photos_qty. Dominio esperado: inteiro nao negativo; pode ser nulo.",

        "peso_gramas":
        "Peso cadastrado do produto em gramas. Origem: product_weight_g. Dominio esperado: decimal nao negativo; pode ser nulo.",

        "comprimento_cm":
        "Comprimento cadastrado do produto em centimetros. Origem: product_length_cm. Dominio esperado: decimal nao negativo; pode ser nulo.",

        "altura_cm":
        "Altura cadastrada do produto em centimetros. Origem: product_height_cm. Dominio esperado: decimal nao negativo; pode ser nulo.",

        "largura_cm":
        "Largura cadastrada do produto em centimetros. Origem: product_width_cm. Dominio esperado: decimal nao negativo; pode ser nulo."

    },

    # =====================================================
    # DIMENSÃO DE TEMPO
    # =====================================================

    "dim_tempo": {

        "data_id":
        "Chave da dimensao temporal no formato YYYYMMDD. Origem: data de compra dos pedidos. Dominio: inteiro representando uma data valida.",

        "data":
        "Data calendario utilizada nas analises temporais. Origem: intervalo das datas de compra dos pedidos. Dominio: data valida.",

        "ano":
        "Ano calendario da data. Origem: extracao do ano da coluna data. Dominio: inteiro de quatro digitos.",

        "mes":
        "Mes calendario da data. Origem: extracao do mes da coluna data. Dominio: inteiros de 1 a 12.",

        "trimestre":
        "Trimestre calendario da data. Origem: calculo a partir da coluna data. Dominio: inteiros de 1 a 4.",

        "dia":
        "Dia do mes. Origem: extracao do dia da coluna data. Dominio: inteiros de 1 a 31.",

        "dia_semana":
        "Dia da semana representado numericamente. Origem: calculo a partir da coluna data. Dominio: inteiros de 1 a 7, conforme convencao utilizada na geracao da dimensao.",

        "ano_mes":
        "Representacao textual do ano e mes para agrupamentos temporais. Origem: formatacao da coluna data. Dominio: formato YYYY-MM."

    },

    # =====================================================
    # DIMENSÃO DE VENDEDORES
    # =====================================================

    "dim_vendedores": {

        "seller_id":
        "Identificador unico do vendedor. Origem: silver.sellers.seller_id. Dominio: identificador textual.",

        "cep_prefixo":
        "Prefixo de cinco digitos do CEP do vendedor. Origem: seller_zip_code_prefix. Dominio esperado: cinco digitos.",

        "cidade":
        "Cidade de localizacao do vendedor. Origem: seller_city. Dominio: nome textual de municipio brasileiro.",

        "estado":
        "Unidade federativa do vendedor. Origem: seller_state. Dominio: siglas das 27 unidades federativas brasileiras."

    },

    # =====================================================
    # FATO DE ITENS DE PEDIDO
    # =====================================================

    "fato_itens_pedido": {

        "order_id":
        "Identificador do pedido ao qual o item pertence. Origem: silver.order_items.order_id. Relacionamento com fato_pedidos.",

        "order_item_id":
        "Numero sequencial do item dentro do pedido. Origem: silver.order_items.order_item_id. Dominio esperado: inteiro positivo.",

        "product_id":
        "Identificador do produto comercializado. Origem: silver.order_items.product_id. Relacionamento com dim_produtos.",

        "seller_id":
        "Identificador do vendedor responsavel pelo item. Origem: silver.order_items.seller_id. Relacionamento com dim_vendedores.",

        "data_id":
        "Chave temporal correspondente a data de compra do pedido. Origem: fato_pedidos.data_id. Relacionamento com dim_tempo.",

        "valor_produto":
        "Preco do produto no item do pedido, em reais. Origem: silver.order_items.price. Dominio esperado: decimal nao negativo.",

        "valor_frete":
        "Valor do frete associado ao item, em reais. Origem: silver.order_items.freight_value. Dominio esperado: decimal nao negativo.",

        "valor_total_item":
        "Valor comercial total do item, incluindo produto e frete, em reais. Transformacao: valor_produto + valor_frete. Dominio esperado: decimal nao negativo."

    },

    # =====================================================
    # FATO DE PAGAMENTOS
    # =====================================================

    "fato_pagamentos": {

        "order_id":
        "Identificador do pedido associado ao pagamento. Origem: silver.order_payments.order_id. Relacionamento com fato_pedidos.",

        "payment_sequential":
        "Numero sequencial do registro de pagamento dentro do pedido. Origem: silver.order_payments.payment_sequential. Dominio esperado: inteiro positivo.",

        "data_id":
        "Chave temporal correspondente a data de compra do pedido. Origem: fato_pedidos.data_id. Relacionamento com dim_tempo.",

        "payment_type":
        "Meio de pagamento utilizado. Origem: silver.order_payments.payment_type. Dominio observado: credit_card, boleto, voucher, debit_card e not_defined.",

        "payment_installments":
        "Quantidade de parcelas registrada no pagamento. Origem: silver.order_payments.payment_installments. Dominio observado: inteiros de 0 a 24.",

        "payment_value":
        "Valor financeiro registrado no pagamento, em reais. Origem: silver.order_payments.payment_value. Dominio esperado: decimal nao negativo."

    },

    # =====================================================
    # FATO DE PEDIDOS
    # =====================================================

    "fato_pedidos": {

        "order_id":
        "Identificador unico do pedido. Origem: silver.orders.order_id. Dominio: identificador textual.",

        "customer_id":
        "Identificador do cliente associado ao pedido. Origem: silver.orders.customer_id. Relacionamento com dim_clientes.",

        "data_id":
        "Chave temporal correspondente a data de compra do pedido. Transformacao: formatacao de order_purchase_timestamp como YYYYMMDD. Relacionamento com dim_tempo.",

        "order_status":
        "Situacao operacional do pedido. Origem: silver.orders.order_status. Dominio: categorias de status existentes no dataset Olist.",

        "order_purchase_timestamp":
        "Data e horario de realizacao da compra. Origem: silver.orders.order_purchase_timestamp. Dominio: timestamp valido.",

        "order_delivered_customer_date":
        "Data e horario efetivo da entrega ao consumidor. Origem: silver.orders.order_delivered_customer_date. Dominio: timestamp valido ou nulo quando nao ha data registrada.",

        "order_estimated_delivery_date":
        "Data estimada para entrega do pedido. Origem: silver.orders.order_estimated_delivery_date. Dominio: timestamp valido.",

        "tempo_entrega_dias":
        "Tempo decorrido entre compra e entrega, em dias. Transformacao: diferenca entre timestamps dividida por 86400. Dominio esperado: numero nao negativo ou nulo.",

        "atraso_dias":
        "Quantidade de dias de atraso em relacao a data estimada. Transformacao: maior valor entre zero e a diferenca das datas de entrega e previsao. Dominio: inteiro nao negativo ou nulo.",

        "status_prazo":
        "Classificacao do prazo de entrega. Transformacao: comparacao entre data efetiva e data estimada. Dominio: atrasado, no_prazo ou nao_entregue.",

        "nota_media_avaliacao":
        "Media das notas de avaliacao associadas ao pedido. Origem: silver.order_reviews.review_score, agregada por order_id. Dominio esperado: valores de 1 a 5 ou nulo.",

        "quantidade_avaliacoes":
        "Quantidade de avaliacoes associadas ao pedido. Origem: contagem de registros de silver.order_reviews por order_id. Dominio: inteiro positivo ou nulo para pedidos sem avaliacao.",

        "quantidade_comentarios":
        "Quantidade de avaliacoes com comentario associadas ao pedido. Origem: soma do indicador possui_comentario em silver.order_reviews. Dominio: inteiro nao negativo ou nulo para pedidos sem avaliacao."

    }

}

#### 3.2 Registro dos comentários no Unity Catalog

Execução automatizada dos comandos de documentação
dos atributos das tabelas Gold.

Cada comentário será persistido nos metadados da
respectiva coluna, permitindo sua consulta diretamente
pelo Catalog Explorer do Databricks.

In [0]:
%python
# Registra os comentários das colunas no Unity Catalog

total_documentadas = 0

for tabela, colunas in documentacao.items():

    print(f"\nDocumentando tabela: {tabela}")

    for coluna, descricao in colunas.items():

        # Escapa aspas simples para evitar erros no SQL
        descricao_sql = descricao.replace("'", "''")

        comando_sql = f"""
            ALTER TABLE workspace.gold.`{tabela}`
            ALTER COLUMN `{coluna}`
            COMMENT '{descricao_sql}'
        """

        spark.sql(comando_sql)

        total_documentadas += 1

    print(f"Tabela {tabela} documentada com sucesso!")

print("\n----------------------------------")
print(f"Total de atributos documentados: {total_documentadas}")
print("----------------------------------")


Documentando tabela: dim_clientes
Tabela dim_clientes documentada com sucesso!

Documentando tabela: dim_produtos
Tabela dim_produtos documentada com sucesso!

Documentando tabela: dim_tempo
Tabela dim_tempo documentada com sucesso!

Documentando tabela: dim_vendedores
Tabela dim_vendedores documentada com sucesso!

Documentando tabela: fato_itens_pedido
Tabela fato_itens_pedido documentada com sucesso!

Documentando tabela: fato_pagamentos
Tabela fato_pagamentos documentada com sucesso!

Documentando tabela: fato_pedidos
Tabela fato_pedidos documentada com sucesso!

----------------------------------
Total de atributos documentados: 54
----------------------------------


#### 3.3 Validação da documentação dos atributos

Consulta aos metadados do Unity Catalog para verificar
a quantidade de atributos documentados.

A validação permite identificar eventuais colunas
sem descrição e confirmar a completude do catálogo
de dados da camada Gold.

In [0]:
%sql

SELECT

    table_name,

    COUNT(*) AS total_colunas,

    COUNT(comment) AS colunas_documentadas,

    COUNT(*) - COUNT(comment) AS colunas_sem_documentacao

FROM workspace.information_schema.columns

WHERE table_schema = 'gold'
  AND table_catalog = 'workspace'

GROUP BY table_name

ORDER BY table_name;

table_name,total_colunas,colunas_documentadas,colunas_sem_documentacao
dim_clientes,5,5,0
dim_produtos,10,10,0
dim_tempo,8,8,0
dim_vendedores,4,4,0
fato_itens_pedido,8,8,0
fato_pagamentos,6,6,0
fato_pedidos,13,13,0
